# nb03c: Stage 1 Linear Probe on nb03_best.pt

* * *

**QUESTION:** How much downstream-task signal does z_price from the nb03_best.pt checkpoint carry, relative to an untrained random encoder with the same architecture?

**H1:** Trained z_price linearly decodes future cumulative log-return, future volatility, and/or future return direction with higher quality than untrained z_price by more than sampling noise. The encoder has learned a representation useful for at least one of these targets.

**H0 (null):** Trained z_price probe metrics match untrained-encoder probe metrics within sampling noise on all three targets. The JEPA training added no decodable downstream signal beyond what a random projection provides.

**MEASURED TARGETS:**
- Regression on future_return (sum of close log-returns over horizon=16): val R^2 at best ridge alpha in {0.01, 0.1, 1, 10, 100}. Sign agreement reported.
- Regression on future_volatility (std of close log-returns over horizon=16): val R^2 at best ridge alpha.
- Logistic regression on sign(future_return): val accuracy and val AUROC at best L2 alpha.
- All three vs the same probes on an untrained encoder (fresh `build_components(cfg)`, no checkpoint).
- Test 4 — backbone (pre-projection) probe: future_volatility ridge (and direction logistic) fit on the PRE-head `pooled` hidden state, captured via a forward pre-hook on `price_encoder.head`. Run for both trained and untrained. With freeze_backbone=True the head is the ONLY trained module, so backbone-vs-z_price isolates whether the trained head destroys signal the frozen backbone kept.
- n_train_batches=100, n_val_batches=50 (capped by dataset size; collector stops when loader exhausts). Seed=42.

**DECISION RULE:**
- Δr2_return = trained_val_r2_return - untrained_val_r2_return; signal if Δ > 0.01
- Δr2_vol    = trained_val_r2_vol    - untrained_val_r2_vol;    signal if Δ > 0.01
- Δacc       = trained_val_accuracy  - untrained_val_accuracy;  signal if Δ > 2 * binomial_se (≈ 2*sqrt(0.25/n_val))
- ALL THREE deltas fail → JEPA training added nothing decodable. Pivot to pretrained backbone, do not tune further.
- ANY delta passes → encoder added signal for that target. Inspect which one; record as the first non-degenerate result.
- head_destruction = backbone_val_r2 - zprice_val_r2, computed per model on future_volatility.
  - trained head_destruction >> untrained head_destruction (by > 0.01), or backbone pooled R^2 >> z_price R^2 for the trained model
    → the TRAINED head is the destroyer. A pretrained backbone alone will NOT fix this while the head + JEPA/VICReg loss are unchanged; the loop must change.
  - backbone pooled R^2 ≈ z_price R^2 (both low) → head is fine; the frozen RANDOM backbone is simply a weak extractor → pretrained backbone (TimesFM) is the right pivot.

**PRIORS / ASSUMPTIONS:**
- Stage 0 verdict was 'degenerate' (shuffled/true ratio = 0.999). We expect this probe to confirm by showing trained ≈ untrained on all three metrics. A finding to the contrary would be surprising and worth a second pass.
- Probe targets derived from RAW (unnormalized) target windows; encoder receives per-sample-normalized context (re-normalized in the collector to match training distribution exactly).
- Direction class balance is approximately 50/50 (close log-returns are roughly symmetric at short horizons); majority-class baseline ≈ 0.5.
- Volatility is autocorrelated and typically easier to predict than direction; if even vol is undecodable, the representation is empty.
- Probe is a contract: ridge alphas, target definitions, and normalization are fixed. Do not silently re-tune them between runs without bumping the diagnostic version.

**FALSIFIER:** H1 is falsified if all three trained metrics fail to exceed untrained by the specified margins. Equally: if trained val_r2 is NEGATIVE on both regression targets (worse than predict-train-mean), the representation is anti-informative — a stronger falsification than 'just no signal'.

**RESULT:** *(fill in after running)*

**DECISION + NEXT ACTION:** *(fill in after running)*

* * *


In [ ]:
# == Colab setup (skipped in VS Code / local kernel) ==
import os, sys

IN_COLAB = 'google.colab' in sys.modules
IN_VSCODE = 'VSCODE_PID' in os.environ or 'VSCODE_CWD' in os.environ
if IN_VSCODE:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content
    !git clone https://github.com/shreyasnat2804/JEPA-quant.git 2>/dev/null || (cd JEPA-quant && git pull)
    %cd /content/JEPA-quant
    %pip install -q "transformers>=4.44" "peft>=0.11" accelerate einops matplotlib pyarrow
    # Colab preinstalls torchao 0.10.0. Recent PEFT raises (not returns False) when it
    # finds torchao below its 0.16.0 minimum, blowing up get_peft_model() during LoRA
    # dispatch even for plain LoRA. Remove it; upgrading would drag torch/ABI churn.
    %pip uninstall -q -y torchao

# Add src to path regardless of environment
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..' if 'notebooks' in os.getcwd() else '.'))
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('repo_root:', repo_root)
print('IN_COLAB:', IN_COLAB, '  IN_VSCODE:', IN_VSCODE)


In [ ]:
# == autoreload shim (imp removed in Python 3.12) ==
import types, importlib
if 'imp' not in sys.modules:
    _imp_shim = types.ModuleType('imp')
    _imp_shim.reload = importlib.reload
    sys.modules['imp'] = _imp_shim
%load_ext autoreload
%autoreload 2


In [ ]:
# == user configuration: edit before running ==
import os

# Path to the best checkpoint saved by JEPATrainer (keys: step, val_jepa, price_encoder, predictor, opt)
CHECKPOINT_PATH = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/checkpoints/nb03_best.pt'

# Directory containing per-ticker .parquet files (same as used for training)
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw/stocks'

# Probe data caps. Collector stops early if loader exhausts, so overshooting is safe.
# n_train * batch_size = nominal probe-fit sample count (capped by train split size).
N_TRAIN_BATCHES = 100  # 100 * 256 = 25,600 samples nominal
N_VAL_BATCHES = 50     # 50  * 256 = 12,800 samples nominal

# Ridge / logistic L2 sweep — kept fixed so probe results are comparable across runs.
RIDGE_ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0)
LOG_ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0)

SEED = 42
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print('device:', DEVICE)


In [ ]:
# == build config, load trained checkpoint, build untrained baseline ==
# Architecture matches nb03: d_model=768, n_layers=8, n_heads=12, freeze_backbone=True.
import torch
from jepa_quant.config import (
    JEPAConfig, PriceEncoderConfig, DataConfig, TrainConfig, PredictorConfig,
)
from jepa_quant.eval.diagnostics import load_checkpoint
from jepa_quant.training.trainer import build_components

cfg = JEPAConfig(
    price_encoder=PriceEncoderConfig(
        backend='transformer',
        n_features=6,
        context_length=64,
        d_model=768,
        n_heads=12,
        n_layers=8,
        latent_dim=256,
        freeze_backbone=True,
    ),
    data=DataConfig(
        data_dir=DATA_DIR,
        context_length=64,
        horizon=16,
        val_fraction=0.15,
        normalize=True,
    ),
    train=TrainConfig(
        batch_size=256,
        num_workers=2,
        device=DEVICE,
        seed=SEED,
    ),
)

# Trained: load the nb03 checkpoint
trained = load_checkpoint(CHECKPOINT_PATH, cfg, device=DEVICE)
print(f'trained loaded from {CHECKPOINT_PATH}')

# Untrained: fresh random init, same architecture, no checkpoint
torch.manual_seed(SEED)
untrained = build_components(cfg)
for mod in (untrained.price_encoder, untrained.target_encoder, untrained.predictor, untrained.regularizer):
    mod.to(DEVICE).eval()
print('untrained baseline built (random init, same arch)')


## Test 1: Ridge regression — future cumulative log-return

Target = sum of close log-returns over the H=16 target window (raw, unnormalized).
Decision: trained val_r2 must beat untrained val_r2 by more than 0.01 to count as signal.


In [ ]:
from jepa_quant.eval.linear_probe import linear_probe_regression

probe_args = dict(
    n_train_batches=N_TRAIN_BATCHES,
    n_val_batches=N_VAL_BATCHES,
    ridge_alphas=RIDGE_ALPHAS,
    seed=SEED,
    device=DEVICE,
)

ret_trained = linear_probe_regression(trained, cfg, target_kind='future_return', **probe_args)
ret_untrained = linear_probe_regression(untrained, cfg, target_kind='future_return', **probe_args)

print('=== Regression: future_return ===')
print(f"                       trained    untrained   delta")
print(f"  val R^2:             {ret_trained['val_r2']:+.4f}   {ret_untrained['val_r2']:+.4f}    {ret_trained['val_r2']-ret_untrained['val_r2']:+.4f}")
print(f"  train R^2 (best a):  {ret_trained['train_r2_at_best_alpha']:+.4f}   {ret_untrained['train_r2_at_best_alpha']:+.4f}")
print(f"  val MSE / baseline:  {ret_trained['val_mse_over_baseline']:.4f}     {ret_untrained['val_mse_over_baseline']:.4f}")
print(f"  sign agreement:      {ret_trained['sign_agreement']:.4f}     {ret_untrained['sign_agreement']:.4f}")
print(f"  best alpha:          {ret_trained['best_alpha']:<9}  {ret_untrained['best_alpha']:<9}")
print(f"  n_train / n_val:     {ret_trained['n_train']} / {ret_trained['n_val']}")
print()
delta_ret = ret_trained['val_r2'] - ret_untrained['val_r2']
if delta_ret > 0.01:
    print(f'  PASS: delta val_r2 = {delta_ret:+.4f} > 0.01 — encoder adds return-decoding signal.')
else:
    print(f'  FAIL: delta val_r2 = {delta_ret:+.4f} <= 0.01 — encoder does not add return-decoding signal.')


## Test 2: Ridge regression — future volatility

Target = std of close log-returns over the H=16 target window (raw, unnormalized).
Volatility clusters; this is the most decodable of the three targets. If even this fails, the representation is empty.


In [ ]:
vol_trained = linear_probe_regression(trained, cfg, target_kind='future_volatility', **probe_args)
vol_untrained = linear_probe_regression(untrained, cfg, target_kind='future_volatility', **probe_args)

print('=== Regression: future_volatility ===')
print(f"                       trained    untrained   delta")
print(f"  val R^2:             {vol_trained['val_r2']:+.4f}   {vol_untrained['val_r2']:+.4f}    {vol_trained['val_r2']-vol_untrained['val_r2']:+.4f}")
print(f"  train R^2 (best a):  {vol_trained['train_r2_at_best_alpha']:+.4f}   {vol_untrained['train_r2_at_best_alpha']:+.4f}")
print(f"  val MSE / baseline:  {vol_trained['val_mse_over_baseline']:.4f}     {vol_untrained['val_mse_over_baseline']:.4f}")
print(f"  best alpha:          {vol_trained['best_alpha']:<9}  {vol_untrained['best_alpha']:<9}")
print(f"  n_train / n_val:     {vol_trained['n_train']} / {vol_trained['n_val']}")
print()
delta_vol = vol_trained['val_r2'] - vol_untrained['val_r2']
if delta_vol > 0.01:
    print(f'  PASS: delta val_r2 = {delta_vol:+.4f} > 0.01 — encoder adds volatility-decoding signal.')
else:
    print(f'  FAIL: delta val_r2 = {delta_vol:+.4f} <= 0.01 — encoder does not add volatility-decoding signal.')


## Test 3: Logistic regression — future return direction

Target = sign(future cumulative log-return). Binary classification. Reports accuracy + AUROC. Threshold for 'signal' uses binomial standard error.


In [ ]:
from jepa_quant.eval.linear_probe import linear_probe_direction
import math

dir_args = dict(
    n_train_batches=N_TRAIN_BATCHES,
    n_val_batches=N_VAL_BATCHES,
    alphas=LOG_ALPHAS,
    seed=SEED,
    device=DEVICE,
)

dir_trained = linear_probe_direction(trained, cfg, **dir_args)
dir_untrained = linear_probe_direction(untrained, cfg, **dir_args)

n_val = dir_trained['n_val']
binom_se = math.sqrt(0.25 / n_val)
threshold = 2 * binom_se

print('=== Direction: sign(future_return) ===')
print(f"                       trained    untrained   delta")
print(f"  val accuracy:        {dir_trained['val_accuracy']:.4f}     {dir_untrained['val_accuracy']:.4f}     {dir_trained['val_accuracy']-dir_untrained['val_accuracy']:+.4f}")
print(f"  val AUROC:           {dir_trained['val_auroc']:.4f}     {dir_untrained['val_auroc']:.4f}     {dir_trained['val_auroc']-dir_untrained['val_auroc']:+.4f}")
print(f"  train acc (best a):  {dir_trained['train_accuracy_at_best_alpha']:.4f}     {dir_untrained['train_accuracy_at_best_alpha']:.4f}")
print(f"  majority baseline:   {dir_trained['majority_class_accuracy']:.4f}     (positive class rate = {dir_trained['positive_class_rate_val']:.4f})")
print(f"  best alpha:          {dir_trained['best_alpha']:<9}  {dir_untrained['best_alpha']:<9}")
print(f"  n_val / binom_se:    {n_val} / {binom_se:.4f}  (threshold for signal = 2*se = {threshold:.4f})")
print()
delta_acc = dir_trained['val_accuracy'] - dir_untrained['val_accuracy']
if delta_acc > threshold:
    print(f'  PASS: delta acc = {delta_acc:+.4f} > {threshold:.4f} — encoder adds direction signal.')
else:
    print(f'  FAIL: delta acc = {delta_acc:+.4f} <= {threshold:.4f} — encoder does not add direction signal.')
# Also flag if trained doesn't even beat majority class
if dir_trained['val_accuracy'] <= dir_trained['majority_class_accuracy']:
    print('  EXTRA FAIL: trained does not beat majority-class baseline. Probe is at chance level.')


## Test 4: Backbone (pre-projection) probe — head vs. backbone

Under `freeze_backbone=True` (the nb03 config) the `ProjectionHead` is the **only** trained
module in the encoder: `input_proj`, the CLS token, and the `nn.TransformerEncoder` backbone
are all frozen random init. Forward is
`input_proj(x) → prepend cls → +pos → backbone → pooled = h[:,0] → head(pooled) → z_price`.

So if trained `z_price` decodes a target **worse** than untrained `z_price` (the anti-informative
result on val), the trained head is the only thing that could have caused it. This test probes the
pre-head `pooled` (the frozen-backbone representation, captured via a forward pre-hook on
`price_encoder.head`) and compares it to the post-head `z_price` probe from Test 2:

- **backbone R² ≫ z_price R² (trained)** → the trained head destroys signal the frozen backbone
  preserved. A pretrained backbone alone won't help while head + loss are unchanged.
- **backbone R² ≈ z_price R² (both low)** → head is fine; the frozen random backbone is a weak
  extractor → pretrained backbone (TimesFM) is the right pivot.

Untrained is run the same way as a reference for how much a **random** head destroys, so we can
tell whether the trained head destroys *more* than random.


In [ ]:
# == Test 4: backbone (pre-projection) probe — future_volatility (clearest target) ==
# probe_source='backbone' taps the input to price_encoder.head via a forward pre-hook.
# z_price numbers reuse Test 2 (vol_trained / vol_untrained) — identical probe, post-head.
bb_args = dict(probe_args, probe_source='backbone')

vol_bb_trained = linear_probe_regression(trained, cfg, target_kind='future_volatility', **bb_args)
vol_bb_untrained = linear_probe_regression(untrained, cfg, target_kind='future_volatility', **bb_args)

print('=== Test 4: future_volatility val R^2 — {trained, untrained} x {backbone pooled, z_price} ===')
print(f"  representation        trained      untrained")
print(f"  backbone pooled       {vol_bb_trained['val_r2']:+.4f}      {vol_bb_untrained['val_r2']:+.4f}    (D={vol_bb_trained['latent_dim']})")
print(f"  z_price (post-head)   {vol_trained['val_r2']:+.4f}      {vol_untrained['val_r2']:+.4f}    (D={vol_trained['latent_dim']})")
print()
# head_destruction = backbone_val_r2 - zprice_val_r2 : how much signal the head discards.
hd_trained = vol_bb_trained['val_r2'] - vol_trained['val_r2']
hd_untrained = vol_bb_untrained['val_r2'] - vol_untrained['val_r2']
print('  head_destruction (backbone_val_r2 - zprice_val_r2):')
print(f"    trained:    {hd_trained:+.4f}")
print(f"    untrained:  {hd_untrained:+.4f}   (random-head reference)")
print()
if hd_trained > hd_untrained + 0.01:
    print('  -> TRAINED head destroys MORE than a random head: the trained head is the destroyer.')
    print('     A pretrained backbone alone will NOT fix this while head + JEPA/VICReg loss are unchanged.')
elif vol_bb_trained['val_r2'] > vol_trained['val_r2'] + 0.01:
    print('  -> backbone pooled >> z_price (trained): the head discards usable signal. Redesign the head/loop.')
else:
    print('  -> backbone pooled ~= z_price: head is not the bottleneck; the frozen random backbone is')
    print('     simply weak -> pretrained backbone (TimesFM first, no numpy<2 dep) is the right pivot.')

# Optional: direction (sign of future_return) on the backbone representation.
dir_bb_trained = linear_probe_direction(trained, cfg, **dict(dir_args, probe_source='backbone'))
dir_bb_untrained = linear_probe_direction(untrained, cfg, **dict(dir_args, probe_source='backbone'))
print()
print('  --- direction sign(future_return) on backbone pooled ---')
print(f"  backbone val acc:    trained {dir_bb_trained['val_accuracy']:.4f}   untrained {dir_bb_untrained['val_accuracy']:.4f}   (majority {dir_bb_trained['majority_class_accuracy']:.4f})")
print(f"  backbone val AUROC:  trained {dir_bb_trained['val_auroc']:.4f}   untrained {dir_bb_untrained['val_auroc']:.4f}")


## Summary and Decision

Fill in the RESULT and DECISION fields in the header cell after running all three tests.

* * *

### Quick reference: decision tree

```
Δr2_return > 0.01  OR  Δr2_vol > 0.01  OR  Δacc > 2*binom_se
  -> encoder adds decodable signal for at least one target. Record which target(s)
     and proceed to inspect representations (compare PCA, retrieve closest-z
     samples, qualitative analysis) before committing to a training pivot.

ALL three deltas fail
  -> JEPA training added no decodable downstream signal beyond a random projection.
     This confirms the Stage 0 'degenerate' verdict from a different angle.
     Pivot to a pretrained backbone (TimesFM first — no numpy<2 dep; Moirai only
     in a dedicated numpy<2 env). Do NOT continue tuning lambda_v / lambda_c on
     the random-frozen-backbone architecture.

Trained val_r2 < 0 on both regression targets
  -> The probe is worse than predicting train-mean on val. The encoder is not
     just empty — its outputs are actively misaligned with downstream signal on
     val. Investigate train/val distribution shift in addition to the pivot.
```

### Reporting in the project log

Quote at least these three numbers when updating CLAUDE.md / research_log:

- `Δval_r2_return` (trained - untrained)
- `Δval_r2_vol` (trained - untrained)
- `Δval_accuracy` (trained - untrained) and `n_val` so the binomial SE is reproducible
- `head_destruction` on future_volatility for both trained and untrained (= backbone_val_r2 - zprice_val_r2), to attribute any anti-informative result to the head vs. the backbone


In [ ]:
# == consolidated results dict for log entry ==
import json, math

n_val = dir_trained['n_val']
binom_se = math.sqrt(0.25 / n_val)
threshold_acc = 2 * binom_se

results_summary = {
    'checkpoint': CHECKPOINT_PATH,
    'n_train_batches': N_TRAIN_BATCHES,
    'n_val_batches': N_VAL_BATCHES,
    'seed': SEED,
    'future_return': {
        'trained_val_r2': ret_trained['val_r2'],
        'untrained_val_r2': ret_untrained['val_r2'],
        'delta_val_r2': ret_trained['val_r2'] - ret_untrained['val_r2'],
        'trained_sign_agreement': ret_trained['sign_agreement'],
        'trained_best_alpha': ret_trained['best_alpha'],
        'passes_decision_rule': (ret_trained['val_r2'] - ret_untrained['val_r2']) > 0.01,
    },
    'future_volatility': {
        'trained_val_r2': vol_trained['val_r2'],
        'untrained_val_r2': vol_untrained['val_r2'],
        'delta_val_r2': vol_trained['val_r2'] - vol_untrained['val_r2'],
        'trained_best_alpha': vol_trained['best_alpha'],
        'passes_decision_rule': (vol_trained['val_r2'] - vol_untrained['val_r2']) > 0.01,
    },
    'direction': {
        'trained_val_accuracy': dir_trained['val_accuracy'],
        'untrained_val_accuracy': dir_untrained['val_accuracy'],
        'delta_val_accuracy': dir_trained['val_accuracy'] - dir_untrained['val_accuracy'],
        'trained_val_auroc': dir_trained['val_auroc'],
        'untrained_val_auroc': dir_untrained['val_auroc'],
        'majority_baseline': dir_trained['majority_class_accuracy'],
        'binom_se': binom_se,
        'threshold': threshold_acc,
        'passes_decision_rule': (dir_trained['val_accuracy'] - dir_untrained['val_accuracy']) > threshold_acc,
    },
    'backbone_volatility': {
        'trained_backbone_val_r2': vol_bb_trained['val_r2'],
        'untrained_backbone_val_r2': vol_bb_untrained['val_r2'],
        'trained_zprice_val_r2': vol_trained['val_r2'],
        'untrained_zprice_val_r2': vol_untrained['val_r2'],
        'head_destruction_trained': vol_bb_trained['val_r2'] - vol_trained['val_r2'],
        'head_destruction_untrained': vol_bb_untrained['val_r2'] - vol_untrained['val_r2'],
        'backbone_latent_dim': vol_bb_trained['latent_dim'],
        'trained_backbone_val_accuracy': dir_bb_trained['val_accuracy'],
        'untrained_backbone_val_accuracy': dir_bb_untrained['val_accuracy'],
    },
}

results_summary['overall_verdict'] = (
    'signal' if any(t['passes_decision_rule'] for t in (
        results_summary['future_return'],
        results_summary['future_volatility'],
        results_summary['direction'],
    )) else 'no_signal'
)

print(json.dumps(results_summary, indent=2))
